<a href="https://colab.research.google.com/github/KrasnovaOV/A-B-test/blob/main/%D0%94%D0%97_%D0%BA%D0%B0%D0%BA_%D0%BF%D1%80%D0%B8%D0%BD%D1%8F%D1%82%D1%8C_%D1%80%D0%B5%D1%88%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BE_%D0%BF%D1%80%D0%BE%D0%B4%D1%83%D0%BA%D1%82%D0%BE%D0%B2%D1%8B%D1%85_%D0%B8%D0%B7%D0%BC%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D1%8F%D1%85.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Задание:***
1. Выберите топ-3 гипотез из предыдущего ДЗ.
2. Продумайте, какие А/Б тесты можно провести, чтобы проверить эти гипотезы.
3. Для каждого А/Б теста четко сформулируйте:
- тестируемое изменение
- бизнес-цель
- выбранную генеральную совокупность
- гипотезу (какие метрики должны измениться), укажите первичные и вторичные метрики
- метод агрегации значения метрики (поведение среднего юзера vs общее значение метрики)
4. Оформите результат в виде таблицы. Шаблон находится в грануле Тестовые датасеты и шаблонКак работать с файлом для выполнения домашнего задания #2

Результат ДЗ предоставить в виде: ссылки на таблицу в Google Docs или Google таблицах

In [3]:
import pandas as pd

# Топ-3 гипотезы из предыдущего ДЗ на основе ICE Score:
# 1. Повторный заказ (№5) - ICE: 384
# 2. Персональный промокод для RFM (№2) - ICE: 320
# 3. Прозрачная причина Surge (№1) - ICE: 315

hypotheses = [
    {
        '№ п/п': 1,
        'Гипотеза': 'Если добавить кнопку "Заказать эту же поездку снова" для повторяющихся маршрутов (Distance < 5 км), конверсия Price_seen -> Order_made вырастет на 8% за счет сокращения шагов.',
        'Тестируемое изменение': 'Добавление кнопки быстрого повтора в историю поездок и на главный экран (для тех, кто ездит по одному маршруту). При нажатии маршрут и адрес подставляются автоматически.',
        'Бизнес-цель': 'Увеличение частоты заказов (Frequency) и рост количества поездок (Rides) от текущей активной базы без привлечения новых пользователей.',
        'Генеральная совокупность': 'Активные пользователи (сделавшие >= 1 заказа за последние 30 дней), у которых в истории есть поездка Distance < 5 км, совершенная в последние 7 дней.',
        'Первичные метрики': '1. CR (Price_seen -> Order_made) в тестовой группе. 2. Количество поездок на пользователя (Rides per User) за 14 дней.',
        'Вторичные метрики': '1. Retention (DAU/MAU). 2. Средний чек (чтобы убедиться, что мы не просто режем цену, а растим частоту). 3. CSI/NPS (удобство).',
        'Метод агрегации метрик': 'Поведение среднего юзера (Mean). Сравниваем среднее количество поездок на одного пользователя в контрольной и тестовой группах, чтобы исключить влияние "китов" (очень активных юзеров).'
    },
    {
        '№ п/п': 2,
        'Гипотеза': 'Пользователи сегмента RFM "low" (50 436 чел.), при получении пуш-уведомления с персональным промокодом на их следующую короткую дистанцию (<3 км) увеличат Retention (возврат) на 10%.',
        'Тестируемое изменение': 'Рассылка пуш-уведомления с текстом "Скучаем! Поездка до 3 км за полцены" и применением промокода в один клик (deep link сразу в экран заказа с введенным адресом).',
        'Бизнес-цель': 'Реактивация "спящих" пользователей и предотвращение их оттока (Churn prevention). Повышение LTV сегмента "low".',
        'Генеральная совокупность': 'Пользователи сегмента RFM "low", которые не совершали поездок более 14 дней, но открывали приложение хотя бы раз за последние 30 дней.',
        'Первичные метрики': '1. CR (из клика по пушу в Order_made). 2. Retention Rate (доля вернувшихся в течение 7 дней после пуша).',
        'Вторичные метрики': '1. Unsubscribe Rate (отписки от пушей). 2. Доля сгоревших промокодов (не воспользовались). 3. Средний чек (не падает ли маржинальность).',
        'Метод агрегации метрик': 'Общее значение метрики (Absolute / Sum) для оценки бизнес-эффекта + Поведение среднего юзера для оценки качества сегмента. Основной фокус на абсолютное количество вернувшихся юзеров.'
    },
    {
        '№ п/п': 3,
        'Гипотеза': 'Если показывать прозрачную причину Surge (например, "Дождь в районе X, +20% к цене") вместо просто иконки молнии, конверсия из Price_seen в Order_made вырастет на 15%.',
        'Тестируемое изменение': 'Замена статичной иконки Surge на текстовый бейдж с причиной (погода/концерт/час-пик) и точным процентом надбавки поверх стандартного UI карточки цены.',
        'Бизнес-цель': 'Снижение процента отмен (Cancellation Rate) после создания заказа и повышение конверсии в успешную оплату (GMV).',
        'Генеральная совокупность': 'Все пользователи, находящиеся в зонах, где в данный момент включен Surge (независимо от их RFM или возраста).',
        'Первичные метрики': '1. CR (Price_seen -> Order_made). 2. Cancellation Rate (User_cancelled / Order_made) после создания заказа.',
        'Вторичные метрики': '1. Время принятия решения (Time to Order после Price_seen). 2. Количество обращений в поддержку по вопросам цены.',
        'Метод агрегации метрик': 'Поведение среднего юзера (Mean conversion rate). Важно смотреть именно на относительное изменение конверсии, так как погодный Surge — событие неравномерное.'
    }
]

df_ab_tests = pd.DataFrame(hypotheses)

# Сохранение в файл для переноса в Excel
df_ab_tests.to_excel('ab_tests_design.xlsx', index=False)

print("Таблица с дизайном А/Б тестов сформирована.")
print(df_ab_tests[['№ п/п', 'Гипотеза', 'Тестируемое изменение', 'Бизнес-цель']])

Таблица с дизайном А/Б тестов сформирована.
   № п/п                                           Гипотеза  \
0      1  Если добавить кнопку "Заказать эту же поездку ...   
1      2  Пользователи сегмента RFM "low" (50 436 чел.),...   
2      3  Если показывать прозрачную причину Surge (напр...   

                               Тестируемое изменение  \
0  Добавление кнопки быстрого повтора в историю п...   
1  Рассылка пуш-уведомления с текстом "Скучаем! П...   
2  Замена статичной иконки Surge на текстовый бей...   

                                         Бизнес-цель  
0  Увеличение частоты заказов (Frequency) и рост ...  
1  Реактивация "спящих" пользователей и предотвра...  
2  Снижение процента отмен (Cancellation Rate) по...  
